# Notebook 3: Model Training
Train three models — baseline, XGBoost, and LSTM autoencoder — and log everything to MLflow.

## 1. Setup & Load Features

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})

from src.features.feature_pipeline import load_features
data = load_features('FD001')
train_df = data['train_df']
feature_cols = data['feature_cols']
print(f"Train: {len(train_df):,} rows, {len(feature_cols)} features")
print(f"Positive rate (label_warning): {train_df['label_warning'].mean():.2%}")

## 2. Time-Series Cross Validation
⚠️ Always split by TIME in PdM — never random. Random split leaks future information.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.patches as mpatches

train_sorted = train_df.sort_values(['unit_id', 'cycle']).reset_index(drop=True)
tscv = TimeSeriesSplit(n_splits=4)

fig, ax = plt.subplots(figsize=(12, 4))
for fold_idx, (tr_idx, val_idx) in enumerate(tscv.split(train_sorted)):
    ax.barh(fold_idx, len(tr_idx), color='#378ADD', alpha=0.7, height=0.6)
    ax.barh(fold_idx, len(val_idx), left=len(tr_idx), color='#E24B4A', alpha=0.7, height=0.6)

ax.set_yticks(range(4))
ax.set_yticklabels([f'Fold {i+1}' for i in range(4)])
ax.set_xlabel('Samples')
ax.set_title('Time-Series Cross Validation Splits')
blue_p = mpatches.Patch(color='#378ADD', alpha=0.7, label='Training')
red_p = mpatches.Patch(color='#E24B4A', alpha=0.7, label='Validation')
ax.legend(handles=[blue_p, red_p])
plt.tight_layout()
plt.show()
print("Each fold trains on past data, validates on future data — no leakage.")

## 3. Model 1: Rule-Based Threshold Baseline

In [ ]:
from src.models.baseline import ThresholdModel
from src.models.xgboost_model import time_series_cv_evaluate

def split_by_units(df, val_fraction=0.2):
    units = df['unit_id'].unique()
    n_val = max(1, int(len(units) * val_fraction))
    val_units = set(units[:n_val])
    return df[~df['unit_id'].isin(val_units)], df[df['unit_id'].isin(val_units)]

train_sub, val_df = split_by_units(train_df)

baseline = ThresholdModel(zscore_threshold=2.5)
baseline.fit(train_sub[feature_cols])
baseline_metrics = baseline.evaluate(val_df[feature_cols], val_df['label_warning'])

print("=== Rule-Based Baseline ===")
for k, v in baseline_metrics.items():
    if isinstance(v, float):
        print(f"  {k:<25} {v:.4f}")

## 4. Model 2: XGBoost Classifier

In [ ]:
from src.models.xgboost_model import XGBoostPdMClassifier, XGBoostRULRegressor
import mlflow

mlflow.set_experiment('PdM_Rig_Failure_Prediction')

with mlflow.start_run(run_name='xgboost_notebook'):
    clf = XGBoostPdMClassifier(params={'n_estimators': 200})
    clf.fit(train_sub, train_sub['label_warning'], feature_cols, val_df, val_df['label_warning'])
    
    threshold = clf.tune_threshold_for_recall(val_df, val_df['label_warning'], target_recall=0.88)
    metrics = clf.evaluate(val_df, val_df['label_warning'])
    
    mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, float)})
    mlflow.log_param('threshold', threshold)

print("=== XGBoost Classifier ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<35} {v:.4f}")
print(f"  {'estimated_business_cost_usd':<35} ${metrics['estimated_business_cost_usd']:,}")

## 5. XGBoost — Feature Importance

In [ ]:
import shap

shap_df = clf.explain(val_df, max_samples=300)
top_features = shap_df.abs().mean().nlargest(15).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Feature importance
importances = pd.Series(clf.model.feature_importances_, index=feature_cols).nlargest(15)
importances.sort_values().plot(kind='barh', ax=axes[0], color='#378ADD')
axes[0].set_title('XGBoost Feature Importances (top 15)')
axes[0].set_xlabel('Importance score')

# SHAP mean |value|
shap_df[top_features].abs().mean().sort_values().plot(kind='barh', ax=axes[1], color='#EF9F27')
axes[1].set_title('SHAP Mean |Value| (top 15)')
axes[1].set_xlabel('Mean |SHAP value|')

plt.tight_layout()
plt.show()

## 6. Model 3: LSTM Autoencoder

In [ ]:
from src.models.lstm_autoencoder import (
    LSTMAutoencoderTrainer, SensorWindowDataset
)
from torch.utils.data import DataLoader

# Use normalized sensor cols only
lstm_cols = [c for c in feature_cols if '_normalized' in c][:12]
print(f"LSTM feature columns: {len(lstm_cols)}")

# Healthy-only dataset (RUL > 80)
healthy_train = train_sub[train_sub['rul'] > 80]
healthy_val = val_df[val_df['rul'] > 80]

train_ds = SensorWindowDataset(healthy_train, lstm_cols, window_size=30, healthy_only=True)
val_ds = SensorWindowDataset(healthy_val, lstm_cols, window_size=30, healthy_only=True)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(f"Training windows: {len(train_ds):,}")
print(f"Validation windows: {len(val_ds):,}")

trainer = LSTMAutoencoderTrainer(n_features=len(lstm_cols), window_size=30)
trainer.feature_cols = lstm_cols
trainer.fit(train_loader, val_loader, epochs=30, lr=1e-3, patience=8, verbose=True)

threshold = trainer.calibrate_threshold(
    DataLoader(train_ds, batch_size=64, shuffle=False), percentile=95.0
)
print(f"\nAnomaly threshold: {threshold:.6f}")

## 7. LSTM — Training Curve & Anomaly Scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(trainer.history['train_loss'], label='Train', color='#378ADD')
if trainer.history.get('val_loss'):
    axes[0].plot(trainer.history['val_loss'], label='Validation', color='#D85A30')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('LSTM Autoencoder Training Loss')
axes[0].legend()

# Anomaly score vs RUL
all_ds = SensorWindowDataset(train_df, lstm_cols, window_size=30, healthy_only=False)
all_loader = DataLoader(all_ds, batch_size=64, shuffle=False)
scores, _ = trainer.compute_anomaly_scores(all_loader)

window_ruls = []
for _, group in train_df.sort_values(['unit_id','cycle']).groupby('unit_id'):
    vals = group['rul'].values
    for i in range(len(vals) - 30 + 1):
        window_ruls.append(vals[i + 29])
window_ruls = np.array(window_ruls[:len(scores)])

sample_idx = np.random.choice(len(scores), min(2000, len(scores)), replace=False)
axes[1].scatter(window_ruls[sample_idx], scores[sample_idx],
               alpha=0.2, s=5, c=window_ruls[sample_idx], cmap='RdYlGn')
axes[1].axhline(threshold, color='#E24B4A', linestyle='--', label=f'Threshold={threshold:.4f}')
axes[1].set_xlabel('RUL (cycles)')
axes[1].set_ylabel('Reconstruction error (anomaly score)')
axes[1].set_title('LSTM Anomaly Score vs RUL\n(higher score = more abnormal)')
axes[1].legend()

plt.tight_layout()
plt.show()

from scipy.stats import spearmanr
valid = min(len(scores), len(window_ruls))
corr, p = spearmanr(-scores[:valid], window_ruls[:valid])
print(f"Spearman correlation (−score vs RUL): {corr:.4f}  (p={p:.2e})")
print("Negative correlation expected — as RUL decreases, anomaly score increases.")

## 8. Run Full Training Pipeline

In [ ]:
# To train all models and log to MLflow:
# python src/models/train.py

# View all experiments:
# mlflow ui --port 5000

print("Run from project root:")
print("  python src/models/train.py")
print("  mlflow ui --port 5000")